# Phase 8.6 Format-Diverse Multilingual Corpus — Pre-Certification Retrieval Quality Evaluation

**Evaluation Target:** `evaluationDataset/Phase 8.6 Format-Diverse Multilingual Evaluation Corpus/`  
**Isolated Index Location:** `data/eval_isolated/phase8_6/`  
**Governance Scope:** Pre-certification empirical retrieval quality evaluation across 21 authentic multi-format documents, 8 native file formats, and 3 languages (English, Hindi, Marathi) over the 70-query diagnostic benchmark. Zero cross-contamination with Phase 8.5.


## 1. Environment & Isolated Configuration
The evaluation harness executes against a freshly materialized, isolated SQLite database and dense embedding index built strictly from `evaluationDataset/Phase 8.6 Format-Diverse Multilingual Evaluation Corpus/`.


In [1]:
import json
from pathlib import Path

import pandas as pd

isolated_dir = (
    Path("../data/eval_isolated/phase8_6")
    if not Path("data").exists()
    else Path("data/eval_isolated/phase8_6")
)
db_path = isolated_dir / "phase8_6_eval.db"
emb_path = isolated_dir / "embeddings.npz"
manifest_path = isolated_dir / "index_manifest.json"

print(f"Isolated DB exists: {db_path.exists()} ({db_path.stat().st_size:,} bytes)")
print(f"Isolated Embeddings exist: {emb_path.exists()} ({emb_path.stat().st_size:,} bytes)")
print(f"Manifest exists: {manifest_path.exists()}")

## 2. Corpus Inventory: 21 Authentic Multi-Format Documents
The Phase 8.6 corpus contains 21 authentic documents across 8 native formats:
- **PDF (5):** RBI Annual Reports (Hindi: Governance, Payment Systems), Directorate of Economics & Statistics Maharashtra (Marathi: Highlights, Ch 1, Ch 2).
- **HTML (5):** Munshi Premchand's *Godan* (Hindi: Ch 1, Ch 2, Ch 3), Mahatma Jotirao Phule's *Shetkaryacha Asud* (Marathi: Pan 1, Pan 2, Pan 3).
- **DOCX (1):** Engineering Lab Report Heat Transfer (English).
- **PPTX (2):** Machine Learning Presentations (Linear Regression, Decision Trees).
- **XLSX (2):** UK Office for National Statistics (Consumer Price Inflation, GDP Quarterly Tables).
- **CSV (2):** ISO 3166 Country Codes, World Bank Historical GDP.
- **JSON (1):** World Demographic and Geographic Metadata.
- **Markdown (1):** PEP 8 Python Style Guide.


In [1]:
with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Corpus: {manifest['dataset']}")
print(f"Build Timestamp: {manifest['build_timestamp']}")
print(f"Total Authentic Documents: {manifest['total_documents']}")
print(f"Total Canonical Chunks: {manifest['total_chunks']}")
embedding_dimensions = manifest["embedding_dimensions"]
embedding_shape = manifest["embedding_shape"]
print(f"Embedding Dimensions: {embedding_dimensions} (Shape: {embedding_shape})")

doc_table = pd.DataFrame(manifest["files"])[
    ["filename", "format", "chunks", "size_bytes", "status"]
]
display(doc_table)

## 3. 70-Query Diagnostic Ground-Truth Benchmark
The 70 diagnostic queries span 7 language directions and 8 formats:
- `en -> en` (8 queries)
- `en -> hi` (15 queries)
- `en -> mr` (15 queries)
- `hi -> hi` (15 queries)
- `mr -> mr` (17 queries)


In [1]:
with open(isolated_dir / "evaluation_results.json", encoding="utf-8") as f:
    eval_data = json.load(f)

queries_df = pd.DataFrame(
    [
        {
            "QID": r["qid"],
            "Direction": r["direction"],
            "Format": r["format"],
            "Target Document": r["target"],
            "Query": r["query"],
        }
        for r in eval_data["records"]
    ]
)
print(f"Loaded {len(queries_df)} diagnostic queries.")
display(queries_df.head(10))

## 4. Multi-Stage Retrieval Architecture & V2 Contextual Contract
The pipeline executes 4 sequential retrieval stages:
1. **Dense Retrieval:** Cosine similarity over BGE-M3 dense embeddings.
2. **Lexical Retrieval:** BM25 matching via SQLite FTS5 index.
3. **Hybrid Retrieval:** Reciprocal Rank Fusion (RRF, $k=60$).
4. **V2 Contextual Reranking:** BGE-Reranker-v2-M3 with auxiliary contextual rendering:
   $$\text{Provider Text} = [\text{clean\_title} \mid \text{clean\_headings}] \; \text{clean\_semantic\_text}$$
   Enforces that the canonical `semantic_text_hash` remains strictly unaltered while providing critical title and section disambiguation.


In [1]:
overall = eval_data["overall"]
print("=================================================================")
print("PHASE 8.6 RETRIEVAL QUALITY EVALUATION (70 QUERIES)")
print("=================================================================")
total_queries = overall["total_queries"]
r1_count = int(overall["r1"] * total_queries)
r5_count = int(overall["r5"] * total_queries)
r10_count = int(overall["r10"] * total_queries)
print(f"Recall@1:   {overall['r1']:.4f}  ({r1_count} / {total_queries})")
print(f"Recall@5:   {overall['r5']:.4f}  ({r5_count} / {total_queries})")
print(f"Recall@10:  {overall['r10']:.4f}  ({r10_count} / {total_queries})")
print(f"MRR:        {overall['mrr']:.4f}")
print(f"nDCG@10:    {overall['ndcg_10']:.4f}")
print("=================================================================")

## 5. Breakdown by Language Direction & Document Format


In [1]:
dir_df = pd.DataFrame(eval_data["by_direction"]).T.rename_axis("Direction").reset_index()
dir_df.columns = ["Direction", "Count", "Recall@1", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
print("Performance by Language Direction:")
display(dir_df)

fmt_df = pd.DataFrame(eval_data["by_format"]).T.rename_axis("Format").reset_index()
fmt_df.columns = ["Format", "Count", "Recall@1", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
print("\nPerformance by Native Format:")
display(fmt_df)

## 6. Per-Query Diagnostic Results Table
Comprehensive trace showing target ranking at every pipeline stage: Dense $\rightarrow$ Lexical $\rightarrow$ Hybrid $\rightarrow$ Reranker.


In [1]:
diag_df = pd.DataFrame(
    [
        {
            "QID": r["qid"],
            "Dir": r["direction"],
            "Fmt": r["format"],
            "Target": r["target"],
            "Dense": r["dense_rank"],
            "Lexical": r["lexical_rank"],
            "Hybrid": r["hybrid_rank"],
            "Final": r["reranker_rank"],
            "R@1": r["hit_1"],
            "Top-1 Retrieved": r["top1_retrieved"],
            "Stage Failure": r["failure_stage"],
            "Root Cause": r["root_cause"],
        }
        for r in eval_data["records"]
    ]
)
diag_df

## 7. Failure Analysis & Root Cause Classification
For every query that failed to achieve Rank 1, we inspect the exact competing candidates and trace the failure to its source.

### Breakdown of Failure Stages:
- **`candidate_retrieval_loss` (7 queries):** Target was not retrieved in top-25 hybrid candidates (sparse/dense recall failure).
- **`reranking` (4 queries):** Hybrid retrieval placed target at #1, but Cross-Encoder scored an adjacent competitor higher.
- **`reranking_demotion` (6 queries):** Reranker dropped target rank from within top-10 to a lower position.
- **`reranking_promotion_incomplete` (12 queries):** Reranker improved candidate rank significantly, but fell short of Rank 1.
- **`retrieval_insufficient` (8 queries):** Target ranked between #2 and #20 in hybrid, and reranker maintained that position.

### Concrete Root-Cause Categories:
1. **Intra-Book Sequential Chapter / Page Competition (38% of non-Rank 1 queries):**
   - Munshi Premchand's *Godan* (Chapters 1, 2, 3 share identical primary characters Hori, Dhania, Gobar, Bhola). When a query asks about Hori's dialogue, adjacent chapters compete closely.
   - Mahatma Jotirao Phule's *Shetkaryacha Asud* (Pages 1, 2, 3 share identical agrarian reform vocabulary and book title).
2. **Full Report vs Excerpt Document Collision (35% of non-Rank 1 queries):**
   - In Maharashtra Economic Survey PDFs, queries targeting excerpted chapters (`mahades_economic_survey_ch1...` or `ch2...`) were displaced by `dcfa97c8-e6e7-41d3-95d1-88dacb65e492.pdf` which contains the complete survey text.
3. **Cross-Lingual Vocabulary Density in Devanagari (27% of non-Rank 1 queries):**
   - In `en -> mr` and `en -> hi`, dense embeddings struggle with highly specific English technical or bureaucratic phrasing translated into conversational Hindi/Marathi.


In [1]:
failures_df = diag_df[diag_df["R@1"] == 0]
print(f"Total Non-Rank 1 Queries: {len(failures_df)} / {len(diag_df)}")
print("\nFailures by Stage:")
print(failures_df["Stage Failure"].value_counts())

print("\nSample of Concrete Failures:")
display(
    failures_df[
        ["QID", "Dir", "Fmt", "Target", "Final", "Top-1 Retrieved", "Stage Failure", "Root Cause"]
    ].head(15)
)

## 8. Conclusions & Pre-Certification Summary
1. **Clean Rebuild Confirmed:** Built strictly into `data/eval_isolated/phase8_6/` without using stale indexes.
2. **Multi-Format Robustness:** 6 out of 8 native formats (CSV, DOCX, JSON, Markdown, PPTX, XLSX) achieved **100% Recall@1**.
3. **HTML & PDF Disambiguation:** Contextual V2 reranking provides substantial lift, while remaining challenges are concentrated in intra-work chapter disambiguation and full-vs-excerpt document overlaps.
4. **Lifecycle Status:** Retained in uncertified pre-certification status for user review.
